# ChartQA VLM baselines

Runs a base **vision-language** model on the ChartAudit v1.0 **test** split (180 items, never trained on) and scores it.

**Variants** (same items and targets, only the input differs):
- `image_only`: the chart image and the question. The model has to read the chart itself.
- `image_table`: the chart image plus the chart data as text (table and colours), then the question.

**Models** (all load in 4-bit):
- `Qwen/Qwen2.5-VL-3B-Instruct` (default: the model the mentor approved for the shared GPU)
- `Qwen/Qwen2.5-VL-7B-Instruct` (within the Track A ≤8B limit)
- `Qwen/Qwen2-VL-7B-Instruct`
- `meta-llama/Llama-3.2-11B-Vision-Instruct` (gated, 11B so above the ≤8B limit; comparison only, zero-shot only because it takes one image per prompt)

**Runtime:** `Runtime → Change runtime type`. An **L4 or A100** is comfortable. A free **T4** works for the 7B models in 4-bit with `BATCH_SIZE = 2`, but is slow (roughly 30–45 minutes per run on the test split).

**Secrets** (key icon in the left sidebar, turn on "Notebook access"):
- `HF_TOKEN`: a Hugging Face token (needed for Llama; accept its licence on the model page first).

**Metrics**
- **Relaxed accuracy**: numbers within 5% of gold, except answers to year questions, which must match exactly. Text and yes/no after normalising case and punctuation. List answers match item by item in any order. Unanswerable items are correct only when the output has an `answer` key set to null. Rules are in `scripts/scoring.py`.
- **JSON valid / Schema valid**: output parses as JSON / has all five fields with an allowed `answer_type`.
- **Evidence grounded**: share of the model's evidence strings that match a real cell, colour or title in the chart's underlying data (`input_text`). In `image_only` the model never sees that text, so this measures how well it read the chart. The prompt does not state the required evidence format and few-shot examples show it, so **do not compare this across zero-shot, few-shot and fine-tuned runs or report it as a gain**.
- **Answer length**: on the `answer` field, predictions over one word among items with one-word golds, and predictions longer than the gold. Reported separately from accuracy.
- **Truncated**: outputs that hit `MAX_NEW_TOKENS` without an end token.

In [ ]:
# 1. Install
!pip -q install -U "transformers>=4.49" accelerate bitsandbytes pillow

In [ ]:
# 2. Settings
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"  # or "Qwen/Qwen2.5-VL-7B-Instruct", "Qwen/Qwen2-VL-7B-Instruct",
                                          #    "meta-llama/Llama-3.2-11B-Vision-Instruct"
VARIANT = "image_only"   # "image_only" (chart image + question) or "image_table" (image + text table + question)
SPLIT = "test"           # "test" for the reported baseline, "val" while you experiment
N_SHOTS = [0]            # e.g. [0, 2]; each shot adds one more image to the prompt (more memory). Shots come from train.
LIMIT = None             # e.g. 20 for a quick smoke test
BATCH_SIZE = 4           # 2 on a T4
MAX_NEW_TOKENS = 256
LOAD_IN_4BIT = True
MAX_PIXELS = 800 * 800   # larger images are scaled down (keeps visual tokens around 800 per chart for Qwen)
SAVE_TO_DRIVE = True     # results go to MyDrive/chartqa_vlm_baselines/
SEED = 42

In [ ]:
# 3. Get the dataset (clones the public repo)
import os, subprocess
REPO = "milshet/chart-explorer"
ROOT = "/content/chart-explorer"
if not os.path.isdir(f"{ROOT}/data/v1.0"):
    subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{REPO}.git", ROOT], check=True)
DATA_DIR = f"{ROOT}/data/v1.0"
print(sorted(os.listdir(DATA_DIR)))

In [ ]:
# 4. Load the splits
import json, random, sys, collections
from PIL import Image

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

eval_set = read_jsonl(f"{DATA_DIR}/{VARIANT}/{SPLIT}.jsonl")
train = read_jsonl(f"{DATA_DIR}/{VARIANT}/train.jsonl")
if LIMIT:
    eval_set = eval_set[:LIMIT]
sys.path.insert(0, f"{ROOT}/scripts")
import check_teacher as CT   # evidence checker used for the teacher labels

def load_image(rel):
    img = Image.open(f"{DATA_DIR}/{rel}").convert("RGB")
    w, h = img.size
    if w * h > MAX_PIXELS:
        s = (MAX_PIXELS / (w * h)) ** 0.5
        img = img.resize((max(28, int(w * s)), max(28, int(h * s))), Image.BICUBIC)
    return img

print(len(eval_set), "eval items |", VARIANT, "|", collections.Counter(r["answer_type"] for r in eval_set))
load_image(eval_set[0]["image"])

In [ ]:
# 5. Prompts. The eval item's system + user turns are used as-is; few-shot examples are full
#    train conversations (image + question -> grounded JSON answer) placed before it.
def pick_shots(train, k, seed=SEED):
    # One example per answer type first (number, text, yes_no, unanswerable).
    rng = random.Random(seed)
    pool = list(train)
    rng.shuffle(pool)
    order = ["number", "text", "yes_no", "unanswerable", "number", "text"]
    shots, used = [], set()
    for t in order:
        if len(shots) == k:
            break
        for r in pool:
            if r["answer_type"] == t and r["id"] not in used:
                shots.append(r); used.add(r["id"]); break
    return shots

def strip_paths(msg):
    """Chat templates want {"type": "image"} placeholders; the PIL images are passed separately."""
    if isinstance(msg["content"], str):
        return dict(msg)
    return {"role": msg["role"], "content": [{"type": "image"} if c["type"] == "image" else c for c in msg["content"]]}

def build(item, shots):
    """Returns (messages, images) with images in the order their placeholders appear."""
    msgs, images = [strip_paths(item["messages"][0])], []
    for s in shots:
        msgs += [strip_paths(s["messages"][1]), strip_paths(s["messages"][2])]
        images.append(load_image(s["image"]))
    msgs.append(strip_paths(item["messages"][1]))
    images.append(load_image(item["image"]))
    return msgs, images

def to_prompt(processor, msgs):
    try:
        return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except Exception:
        # Templates without a system role: fold the system prompt into the first user turn's text.
        first = {"role": "user", "content": [dict(c) for c in msgs[1]["content"]]}
        for c in first["content"]:
            if c["type"] == "text":
                c["text"] = msgs[0]["content"] + "\n\n" + c["text"]; break
        return processor.apply_chat_template([first] + msgs[2:], tokenize=False, add_generation_prompt=True)

In [ ]:
# 6. Parse and score: the rules live in scripts/scoring.py so every run uses the same ones
from scoring import SCORER_VERSION, score
print("scorer version", SCORER_VERSION)

In [ ]:
# 7. Load the model
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(userdata.get("HF_TOKEN"))
except Exception as e:
    print("HF login skipped:", e)

DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                           bnb_4bit_compute_dtype=DTYPE) if LOAD_IN_4BIT else None
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, device_map="auto", quantization_config=quant,
                                                    torch_dtype=DTYPE)
model.eval()
if "llama" in MODEL_ID.lower() and any(k > 0 for k in N_SHOTS):
    print("Llama 3.2 Vision takes one image per prompt: running zero-shot only.")
    N_SHOTS = [0]
print(MODEL_ID, "loaded on", model.device, "| dtype", DTYPE)

In [ ]:
# 8. Generate (greedy), save the raw outputs, then score
import datetime
from tqdm.auto import tqdm

# Results go to Drive if available, else /content
out_dir = "/content/chartqa_vlm_baselines"
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        out_dir = "/content/drive/MyDrive/chartqa_vlm_baselines"
    except Exception as e:
        print("Drive not mounted, saving locally:", e)
os.makedirs(out_dir, exist_ok=True)
tag = f"{MODEL_ID.split('/')[-1]}_{VARIANT}_{SPLIT}_{datetime.datetime.now():%Y%m%d-%H%M}"

# Record the settings actually used, including defaults inherited from the model
print(model.generation_config)
with open(f"{out_dir}/{tag}_settings.json", "w", encoding="utf-8") as f:
    json.dump({"scorer_version": SCORER_VERSION, "model": MODEL_ID, "variant": VARIANT, "split": SPLIT,
               "n_shots": N_SHOTS, "batch_size": BATCH_SIZE, "max_new_tokens": MAX_NEW_TOKENS,
               "max_pixels": MAX_PIXELS, "load_in_4bit": LOAD_IN_4BIT, "seed": SEED,
               "generation_config": model.generation_config.to_dict()}, f, indent=2, default=str)

eos_ids = model.generation_config.eos_token_id
if eos_ids is None:
    eos_ids = processor.tokenizer.eos_token_id
eos_ids = [eos_ids] if isinstance(eos_ids, int) else list(eos_ids)

@torch.no_grad()
def generate(items, shots):
    """Returns (texts, truncated). An output with no end token hit MAX_NEW_TOKENS."""
    outs, truncated = [], []
    for i in tqdm(range(0, len(items), BATCH_SIZE)):
        built = [build(it, shots) for it in items[i:i + BATCH_SIZE]]
        prompts = [to_prompt(processor, m) for m, _ in built]
        batch = processor(text=prompts, images=[imgs for _, imgs in built], padding=True,
                          return_tensors="pt").to(model.device)
        gen = model.generate(**batch, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             pad_token_id=processor.tokenizer.pad_token_id)
        new = gen[:, batch["input_ids"].shape[1]:]
        outs += processor.batch_decode(new, skip_special_tokens=True)
        stopped = torch.isin(new, torch.tensor(eos_ids, device=new.device)).any(dim=1)
        truncated += (~stopped).tolist()
    return outs, truncated

runs = {}
for k in N_SHOTS:
    shots = pick_shots(train, k)
    raws, truncated = generate(eval_set, shots)
    # Saved before scoring, so a scoring problem cannot lose a run
    with open(f"{out_dir}/{tag}_{k}shot_raw.jsonl", "w", encoding="utf-8") as f:
        for it, raw, t in zip(eval_set, raws, truncated):
            f.write(json.dumps({"id": it["id"], "raw": raw, "truncated": t}, ensure_ascii=False) + "\n")
    runs[k] = [score(it, raw, t) for it, raw, t in zip(eval_set, raws, truncated)]
    acc = sum(r["correct"] for r in runs[k]) / len(runs[k])
    print(f"{VARIANT} | {k}-shot relaxed accuracy: {acc:.3f} | truncated: {sum(truncated)}")
print(runs[N_SHOTS[0]][0]["raw"])

In [ ]:
# 9. Report
import pandas as pd

def summarise(results, items):
    df = pd.DataFrame(results).merge(
        pd.DataFrame([{k: it[k] for k in ("id", "source", "chart_type", "answer_type", "question_type")}
                      for it in items]), on="id")
    answerable = df[df.answer_type != "unanswerable"]
    one_word_gold = answerable[answerable.gold_words == 1]
    overall = {
        "n": len(df),
        "n_answerable": len(answerable),
        "n_refusal": int((df.answer_type == "unanswerable").sum()),
        "relaxed_accuracy": df.correct.mean(),
        "json_valid": df.json_valid.mean(),
        "schema_valid": df.schema_valid.mean(),
        "evidence_grounded": df.evidence_ok.sum() / max(df.evidence_n.sum(), 1),
        "refusal_accuracy": df[df.answer_type == "unanswerable"].correct.mean(),
        "answerable_accuracy": answerable.correct.mean(),
        "one_word_gold_n": len(one_word_gold),
        "over_one_word": int((one_word_gold.pred_words > 1).sum()),
        "longer_than_gold": int((answerable.pred_words > answerable.gold_words).sum()),
        "truncated": int(df.truncated.sum()),
    }
    by = {col: df.groupby(col).correct.agg(["mean", "count"]).round(3) for col in
          ("question_type", "answer_type", "source", "chart_type")}
    return df, overall, by

summary_rows = []
for k, res in runs.items():
    df, overall, by = summarise(res, eval_set)
    summary_rows.append({"model": MODEL_ID, "variant": VARIANT, "split": SPLIT, "shots": k,
                         "scorer_version": SCORER_VERSION,
                         **{a: round(b, 4) for a, b in overall.items()}})
    print(f"\n===== {MODEL_ID} | {VARIANT} | {SPLIT} | {k}-shot =====")
    print(pd.Series(overall).round(3).to_string())
    print(f"note: refusal_accuracy rests on {overall['n_refusal']} items; "
          "evidence_grounded is not comparable across shots or runs")
    for col, t in by.items():
        print(f"\nby {col}:\n{t.to_string()}")
summary = pd.DataFrame(summary_rows)
summary

In [ ]:
# 10. Save predictions and the summary (out_dir and tag are set in cell 8)
for k, res in runs.items():
    with open(f"{out_dir}/{tag}_{k}shot_predictions.jsonl", "w", encoding="utf-8") as f:
        for r in res:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
summary_path = f"{out_dir}/summary.csv"
summary.to_csv(summary_path, mode="a", header=not os.path.exists(summary_path), index=False)
print("saved to", out_dir)

In [ ]:
# 11. Free GPU memory before switching MODEL_ID in cell 2 and re-running cells 7-10
#     (to switch only VARIANT, re-run cells 2, 4, 8-10; the model stays loaded)
import gc
del model; gc.collect(); torch.cuda.empty_cache()